# Task 1: Repository Cloning and File Discovery

### Mục tiêu
- Khám phá toàn bộ kho mã nguồn Python được chọn làm **corpus** cho CPG Parser Service (Task 2).
- Kết quả của task này là danh sách file `.py` đã phân loại, lưu vào `output/discovered_files.csv`. 
  
### Thông tin repo

| Trường        | Giá trị |
|:--------------|:--------|
| **Tên repo**  | `peft` |
| **Tổ chức**   | `huggingface` |
| **URL clone** | `https://github.com/huggingface/peft.git` |


### 0. Imports & cấu hình đường dẫn

In [ ]:
import os
import sys
import subprocess
import platform
from pathlib import Path

import pandas as pd

# Thư mục gốc của project (thư mục chứa notebook)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "peft").exists() and (PROJECT_ROOT.parent / "peft").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Đường dẫn repository và thư mục output trên Windows
WIN_REPO_PATH = str(PROJECT_ROOT / "peft")
WIN_OUTPUT_DIR = str(PROJECT_ROOT / "output")

REPO_CLONE_URL = "https://github.com/huggingface/peft.git"

# Chọn đường dẫn phù hợp với hệ điều hành hiện tại
if platform.system() == "Windows":
    REPO_ROOT = Path(WIN_REPO_PATH)
    OUTPUT_DIR = Path(WIN_OUTPUT_DIR)

# Tạo thư mục output nếu chưa tồn tại
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python     : {sys.version}")
print(f"Hệ điều hành: {platform.system()} - {platform.version()}")
print(f"PROJECT    : {PROJECT_ROOT}")
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

# Thêm thư mục project vào PYTHONPATH để có thể import các module trong src
sys.path.insert(
    0,
    str(Path(os.path.abspath(".")).parent.parent)
    if Path(os.path.abspath(".")).name == "task1" and Path(os.path.abspath(".")).parent.name == "src"
    else str(Path(os.path.abspath(".")))
)

from src.task1.clone import clone_repo_if_needed
from src.task1.discover import (
    is_auto_generated,
    classify_file,
    count_lines,
    scan_repo,
    build_dataframe,
    smoke_test,
)

Python     : 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Hệ điều hành: Windows - 10.0.26200
PROJECT    : E:\BD
REPO_ROOT  : E:\BD\peft
OUTPUT_DIR : E:\BD\output


### 1. Kiểm tra & Clone Repository

In [2]:
clone_repo_if_needed(REPO_ROOT, REPO_CLONE_URL)

assert REPO_ROOT.exists(), f"[LỖI] Không tìm thấy repository tại {REPO_ROOT}"
print(f"[OK] Repository hợp lệ: {REPO_ROOT}")

[INFO] Repository đã tồn tại tại: E:\BD\peft
[INFO] Bỏ qua bước clone.
[OK] Repository hợp lệ: E:\BD\peft


### 2. Duyệt đệ quy & thu thập tất cả file `.py`

In [3]:
all_py_files_relative, categories = scan_repo(REPO_ROOT)

print("\nVí dụ 10 tệp đầu tiên:")
for p in all_py_files_relative[:10]:
    print(f"  {p}")

[OK] Tìm thấy 431 tệp .py

Ví dụ 10 tệp đầu tiên:
  docs\source\_config.py
  examples\adamss_finetuning\glue_adamss_asa_example.py
  examples\adamss_finetuning\glue_adamss_asa_manual_example.py
  examples\adamss_finetuning\image_classification_adamss_asa.py
  examples\adamss_finetuning\test_adamss_quick.py
  examples\alora_finetuning\alora_finetuning.py
  examples\arrow_multitask\arrow_phi3_mini.py
  examples\bdlora_finetuning\chat.py
  examples\beft_finetuning\beft_finetuning.py
  examples\boft_controlnet\__init__.py


### 3. Phân loại file `.py`

**Mục tiêu:** Loại bỏ các file không liên quan trước khi đưa vào CPG Parser, giúp giảm nhiễu
và tăng chất lượng đồ thị phụ thuộc.

**Bốn nhóm phân loại và lý do loại trừ:**

| Nhóm | Tiêu chí | Lý do loại trừ |
|:-----|:---------|:---------------|
| **test** | Thư mục chứa `test`/`tests`; tên file bắt đầu bằng `test_` | Test code không đại diện cho logic nghiệp vụ, chứa mock và fixture gây nhiễu CPG |
| **setup** | Tên file chính xác là `setup.py`, `__init__.py`, `conftest.py` (ngoài thư mục test) | File scaffolding/packaging, không chứa logic domain |
| **auto-generated** | Nằm trong `build/`, `dist/`, `*.egg-info/`; hoặc header chứa `DO NOT EDIT` / `auto-generated` | Code sinh tự động thường bị flatten, không phản ánh cấu trúc thiết kế thực |
| **source** | Tất cả file còn lại | Corpus chính cho CPG Parser Service ở Task 2 |

**Thứ tự ưu tiên phân loại:** `auto-generated` -> `setup` (exact filename) -> `test` (path/prefix) -> `source`

In [4]:
from collections import Counter

category_counts = Counter(categories)

print("=" * 50)
print("KẾT QUẢ PHÂN LOẠI TỆP .py")
print("=" * 50)

for cat in ["source", "test", "setup", "auto-generated"]:
    print(f"  {cat:<20}: {category_counts.get(cat, 0):>5} tệp")

print("-" * 50)
print(f'  {"TỔNG":<20}: {sum(category_counts.values()):>5} tệp')

# Kiểm tra conftest.py phải được phân loại là setup
conftest = [
    (p, c)
    for p, c in zip(all_py_files_relative, categories)
    if p.name == "conftest.py"
]

print("\nKiểm tra conftest.py (phải là setup):")

for path, cat in conftest[:5]:
    status = "OK" if cat == "setup" else "LỖI"
    print(f"  [{status}] {path} -> {cat}")

KẾT QUẢ PHÂN LOẠI TỆP .py
  source              :   309 tệp
  test                :    63 tệp
  setup               :    59 tệp
  auto-generated      :     0 tệp
--------------------------------------------------
  TỔNG                :   431 tệp

Kiểm tra conftest.py (phải là setup):
  [OK] tests\conftest.py -> setup


**Kiểm chứng heuristic is_auto_generated() (Smoke Test)**  
Do repository ban đầu không chứa các file sinh tự động nên kết quả phát hiện là 0 file. Để kiểm tra hàm `is_auto_generated()`, nhóm tạo một số file và thư mục giả lập (ví dụ build/, dist/,...) rồi chạy lại chương trình. Nếu các file này được phát hiện đúng là auto-generated thì có thể kết luận detector hoạt động như mong đợi. Sau khi kiểm tra, nhóm xóa các file giả lập để giữ nguyên trạng thái của repository.

In [5]:
smoke_test()

  [OK] Tệp có 'DO NOT EDIT'
  [OK] Tệp Python thông thường
  [OK] Đường dẫn chứa 'build/'
[OK] Smoke test thành công (3/3)


### 4. Tạo DataFrame & Lưu ra CSV

In [6]:
df = build_dataframe(REPO_ROOT, all_py_files_relative, categories, OUTPUT_DIR)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

print("\n10 dòng đầu:")
df.head(10)

[OK] Đã lưu tệp CSV: E:\BD\output\discovered_files.csv (431 dòng)

10 dòng đầu:


,relative_path,size_bytes,category,num_lines
0,docs/source/_config.py,287,source,7
1,examples/adamss_finetuning/glue_adamss_asa_example.py,15224,source,382
2,examples/adamss_finetuning/glue_adamss_asa_manual_example.py,16825,source,411
3,examples/adamss_finetuning/image_classification_adamss_asa.py,17368,source,452
4,examples/adamss_finetuning/test_adamss_quick.py,4587,test,176
5,examples/alora_finetuning/alora_finetuning.py,9839,source,259
6,examples/arrow_multitask/arrow_phi3_mini.py,14755,source,383
7,examples/bdlora_finetuning/chat.py,1972,source,64
8,examples/beft_finetuning/beft_finetuning.py,4083,source,122
9,examples/boft_controlnet/__init__.py,0,setup,0


In [7]:
summary_df = (
    df.groupby("category")
      .agg(
          num_files=("relative_path", "count"),
          total_bytes=("size_bytes", "sum"),
          total_lines=("num_lines", "sum"),
          avg_lines=("num_lines", "mean"),
      )
      .reset_index()
      .sort_values("num_files", ascending=False)
)

summary_df["avg_lines"] = summary_df["avg_lines"].round(1)

print("Thống kê theo từng loại tệp:")
summary_df

Thống kê theo từng loại tệp:


,category,num_files,total_bytes,total_lines,avg_lines
1,source,309,4052222,93524,302.7
2,test,63,2065517,48096,763.4
0,setup,59,75381,2149,36.4


### 5. Bảng tổng kết cuối cùng (Lab Report)

In [8]:
total_py_files = len(df)
n_test         = category_counts.get('test', 0)
n_setup        = category_counts.get('setup', 0)
n_autogen      = category_counts.get('auto-generated', 0)
n_excluded     = n_test + n_setup + n_autogen
n_source       = category_counts.get('source', 0)

print('╔══════════════════════════════════════════════════════╗')
print('║           TASK 1 – FINAL SUMMARY (Lab Report)        ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Repository : huggingface/peft                       ║')
print(f'║  URL        : https://github.com/huggingface/peft    ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Total .py files discovered        : {total_py_files:>5}           ║')
print('╠──────────────────────────────────────────────────────╣')
print(f'║  Excluded – test files             : {n_test:>5}           ║')
print(f'║  Excluded – setup/init files       : {n_setup:>5}           ║')
print(f'║  Excluded – auto-generated files   : {n_autogen:>5}           ║')
print(f'║  Total excluded                    : {n_excluded:>5}           ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  SOURCE files -> CPG Parser Service : {n_source:>5}          ║')
print('╚══════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════╗
║           TASK 1 – FINAL SUMMARY (Lab Report)        ║
╠══════════════════════════════════════════════════════╣
║  Repository : huggingface/peft                       ║
║  URL        : https://github.com/huggingface/peft    ║
╠══════════════════════════════════════════════════════╣
║  Total .py files discovered        :   431           ║
╠──────────────────────────────────────────────────────╣
║  Excluded – test files             :    63           ║
║  Excluded – setup/init files       :    59           ║
║  Excluded – auto-generated files   :     0           ║
║  Total excluded                    :   122           ║
╠══════════════════════════════════════════════════════╣
║  SOURCE files -> CPG Parser Service :   309          ║
╚══════════════════════════════════════════════════════╝


### 6. Reflection

**What worked**
- `git clone --depth 1` giúp lấy repo `peft` nhanh và nhẹ, đủ dùng cho việc phân tích tĩnh mã nguồn.
- `pathlib.Path.rglob("*.py")` duyệt đệ quy toàn bộ 431 file `.py` mà không cần xử lý riêng cho từng hệ điều hành.
- Heuristic phân loại 4 nhóm (`auto-generated` -> `setup` -> `test` -> `source`) cho kết quả nhất quán, đã kiểm chứng bằng smoke test riêng cho `is_auto_generated()` (3/3 test case PASS).

**What failed**
- Bản đầu tiên đặt thứ tự ưu tiên là `test` trước `setup`, khiến `conftest.py` bị nhận nhầm thành `test` (do `"test"` là substring của `"conftest"`). Notebook đã bổ sung một bước kiểm tra riêng (in ra danh sách `conftest.py` và category tương ứng) để phát hiện lỗi này.
- Repo `peft` không có sẵn file/thư mục auto-generated (`build/`, `dist/`, `.egg-info/`) nên nhóm không thể xác nhận `is_auto_generated()` hoạt động đúng chỉ bằng cách chạy trên repo thật.

**How resolved**
- Đổi thứ tự ưu tiên phân loại: kiểm tra khớp tên file chính xác (`setup`) trước khi kiểm tra từ khoá `test`, giải quyết dứt điểm lỗi `conftest.py`.
- Viết smoke test tạo file/thư mục giả lập (có marker `DO NOT EDIT`, path chứa `build/`) để kiểm chứng `is_auto_generated()` hoạt động đúng, bù lại việc repo thật không có ca auto-generated nào.